# Parte 1 — Análise exploratória com APIs externas

Clima (Open-Meteo), feriados (Nager.Date), padrões geoespaciais e demanda do 1746 (2023–2024).

## Instalando bibliotecas necessárias

## Importando bibliotecas necessárias

In [100]:
import basedosdados as bd
import pandas as pd
import requests

### Configurações

In [101]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Base de dados de chamados

### Baixando dados da base

In [102]:
df_chamados = bd.read_sql(
    "SELECT * FROM `datario.adm_central_atendimento_1746.chamado` WHERE data_particao >= '2023-01-01' AND data_particao <= '2024-12-31' LIMIT 1000",
    billing_project_id="desafio-pic",
)

Downloading: 100%|██████████|


### Análise de colunas 

In [103]:
df_chamados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id_chamado                        1000 non-null   object             
 1   id_origem_ocorrencia              1000 non-null   object             
 2   data_inicio                       1000 non-null   datetime64[us]     
 3   data_fim                          1000 non-null   datetime64[us]     
 4   id_bairro                         1000 non-null   object             
 5   id_territorialidade               1000 non-null   object             
 6   id_logradouro                     1000 non-null   object             
 7   numero_logradouro                 972 non-null    Int64              
 8   id_unidade_organizacional         1000 non-null   object             
 9   nome_unidade_organizacional       1000 non-null   object        

### Análise de dados duplicados
Nenhum registro duplicado foi encontrado

In [104]:
df_chamados.duplicated().sum()

np.int64(0)

### Análise de dados nulos
Foram encontradas as seguintes colunas com dados nulos:
- numero_logradouro: 28
- longitude: 423
- latitude: 28
- data_alvo_diagnostico: 1000
- data_real_diagnostico: 1000
- justificativa_status: 965

In [105]:
print(
    [
        (col, df_chamados[col].isna().sum())
        for col in df_chamados.columns[df_chamados.isna().any()].tolist()
    ]
)

[('numero_logradouro', np.int64(28)), ('longitude', np.int64(423)), ('latitude', np.int64(423)), ('data_alvo_diagnostico', np.int64(1000)), ('data_real_diagnostico', np.int64(1000)), ('justificativa_status', np.int64(965))]


### Análise descritiva dos dados

In [106]:
df_chamados.describe().T

,count,mean,min,25%,50%,75%,max,std
data_inicio,1000,2024-05-17 17:00:58.848000,2024-05-01 12:51:22,2024-05-09 17:02:16.500000,2024-05-20 13:41:10,2024-05-24 14:05:25,2024-05-31 19:52:12,NaN
data_fim,1000,2024-05-24 16:36:08.681999,2024-05-02 12:39:56,2024-05-14 15:44:32,2024-05-24 16:03:25.500000,2024-06-03 08:00:58.250000,2024-07-04 10:07:09,NaN
numero_logradouro,972.0,277.048354,0.0,38.0,103.0,326.5,3680.0,503.95839
longitude,577.0,-43.201524,-47.500442,-43.206911,-43.188838,-43.179153,-43.107261,0.180475
latitude,577.0,-22.888912,-22.988239,-22.924157,-22.910399,-22.90133,-4.943207,0.749171
data_alvo_finalizacao,1000,2024-05-31 09:05:46.320000,2024-05-10 00:00:00,2024-05-22 09:02:30,2024-05-30 02:36:00,2024-06-07 13:25:45,2024-07-11 11:26:00,NaN
data_alvo_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
data_real_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
tempo_prazo,1000.0,9.068,6.0,6.0,7.0,15.0,15.0,3.898535
reclamacoes,1000.0,0.012,0.0,0.0,0.0,0.0,1.0,0.10894


### Tratamento de dados nulos
Considerando as análises iniciais, considerei substituir os valores nulos de:
- colunas Int ou Float pela mediana
- colunas de latitude ou longitude pela moda
- colunas de datas não serão alteradas, pois todos os registros são nulos e até o momento não serão usados
- colunas object não serão alteradas, pois até o momento não serão usadas

#### Coluna numero_logradouro

In [107]:
df_chamados["numero_logradouro"] = df_chamados["numero_logradouro"].fillna(
    df_chamados["numero_logradouro"].median()
)
df_chamados["numero_logradouro"].isna().sum()

np.int64(0)

### Coluna longitude

In [108]:
longitude_moda = df_chamados["longitude"].mode()
if not longitude_moda.empty:
    df_chamados["longitude"] = df_chamados["longitude"].fillna(
        value=float(longitude_moda.iloc[0]),
    )
df_chamados["longitude"].isna().sum()

np.int64(0)

#### Coluna latitude

In [109]:
latitude_moda = df_chamados["latitude"].mode()
if not latitude_moda.empty:
    df_chamados["latitude"] = df_chamados["latitude"].fillna(
        value=float(latitude_moda.iloc[0]),
    )
df_chamados["latitude"].isna().sum()

np.int64(0)

### Tratamento de datas
Para fazer as requisições à API do Open-Meteo é preciso ter a data formatada como "YYYY-MM-dd"

#### Coluna data_inicio

In [110]:
df_chamados["data_inicio"] = pd.to_datetime(df_chamados["data_inicio"]).dt.strftime(
    "%Y-%m-%d"
)
df_chamados["data_inicio"].head()

0    2024-05-04
1    2024-05-14
2    2024-05-23
3    2024-05-28
4    2024-05-08
Name: data_inicio, dtype: object

#### Coluna data_fim

In [111]:
df_chamados["data_fim"] = pd.to_datetime(df_chamados["data_fim"]).dt.strftime(
    "%Y-%m-%d"
)
df_chamados["data_fim"].head()

0    2024-05-13
1    2024-05-14
2    2024-05-24
3    2024-06-03
4    2024-05-13
Name: data_fim, dtype: object

## Baixando dados da API Open-Meteo

## Acessando a API

In [112]:
result = requests.get(
    "https://archive-api.open-meteo.com/v1/archive?latitude=-22.9&longitude=-43.2&start_date=2024-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min&timezone=auto"
)
result.json()

{'latitude': -22.952549,
 'longitude': -43.215027,
 'generationtime_ms': 3.2165050506591797,
 'utc_offset_seconds': -10800,
 'timezone': 'America/Sao_Paulo',
 'timezone_abbreviation': 'GMT-3',
 'elevation': 6.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C'},
 'daily': {'time': ['2024-01-01'],
  'temperature_2m_max': [26.8],
  'temperature_2m_min': [22.3]}}

In [ ]:
from typing import Any


df = df_chamados.copy()

API_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_FIELDS = "temperature_2m_max,temperature_2m_min"
TIMEOUT_SECONDS = 20

# Colunas de saída
for col in ["temperatura_maxima", "temperatura_minima"]:
    df[col] = pd.NA


def is_missing_coords(row: pd.Series) -> bool:
    return bool(pd.isna([row["latitude"], row["longitude"]]).any())


def build_weather_params(row: pd.Series) -> dict[str, Any]:
    return {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": row["data_inicio"],
        "end_date": row["data_fim"],
        "daily": DAILY_FIELDS,
        "timezone": "auto",
    }


def fetch_daily_temperatures(
    session: requests.Session,
    params: dict[str, Any],
) -> tuple[list[Any], list[Any]]:
    response = session.get(API_URL, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()

    daily = response.json().get("daily", {})
    max_values = daily.get("temperature_2m_max", [])
    min_values = daily.get("temperature_2m_min", [])
    return max_values, min_values


session = requests.Session()
for index, row in df.iterrows():
    if is_missing_coords(row):
        continue

    params = build_weather_params(row)

    try:
        max_values, min_values = fetch_daily_temperatures(session, params)
    except requests.RequestException:
        continue

    df.loc[index, "temperatura_maxima"] = max_values[0] if max_values else pd.NA
    df.loc[index, "temperatura_minima"] = min_values[0] if min_values else pd.NA


df[["temperatura_maxima", "temperatura_minima"]].head()